In [1]:
!pip install av==11.0.0
import torch
import torchvision
import os
import re
from google.colab import drive
import math
from torchvision.transforms import v2
import random
! if [ -d pertwee ]; then (cd pertwee; git pull); else git clone https://github.com/occipita/pertwee.git; fi
import pertwee
import importlib
importlib.reload(pertwee) # perwee may have changed due to a "git pull" operation above, so reload it just in case
importlib.reload(pertwee.normalisations)
importlib.reload(pertwee.frameutils)
%matplotlib inline
import matplotlib.pyplot as plt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.9/32.9 MB 60.0 MB/s eta 0:00:00
Cloning into 'pertwee'...
remote: Enumerating objects: 29, done.
remote: Counting objects: 100% (29/29), done.
remote: Compressing objects: 100% (20/20), done.
remote: Total 29 (delta 11), reused 23 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (29/29), 11.41 KiB | 11.42 MiB/s, done.
Resolving deltas: 100% (11/11), done.


In [2]:
#
# global variables that define behaviour
#

# file selection
driveLoc = '/content/drive'
vpath = driveLoc + "/MyDrive/videos"
datapath = driveLoc + "/MyDrive/trainingdata"



In [3]:
if os.path.exists(driveLoc):
  print ("Drive already mounted")
else:
  print ("Mounting drive")
  drive.mount(driveLoc)

trainingData = []
validationData = []

datafiles = os.listdir(datapath)
for f in datafiles:
  fullPath = datapath+"/"+f
  splitext = os.path.splitext(f)
  nameFields = splitext[0].split('-')
  if not os.path.isfile(fullPath) or splitext[1] != ".dat" or len(nameFields) < 3:
    print ("Skipped (not a data set): ",f)
    continue

  if nameFields[0] != "anglechange":
    print ("Skipped (not an angle change data set): ", f)
    continue

  isValidationSet = nameFields[1] == "validation"
  isTrainingSet = nameFields[1] == "training"
  if not isValidationSet and not isTrainingSet:
    print (f"Skipped (unrecognised data set purpose {nameFields[1]}): ",f)
    continue

  print ("Loading: ",f)

  data = torch.load (fullPath, weights_only=True)

  print (f"Successfully loaded {len(data)} {nameFields[1]} items for video file {nameFields[2]}")
  if isValidationSet:
    validationData.extend(data)
  elif isTrainingSet:
    trainingData.extend(data)
  else:
    print ("Don't have anywhere to put this data!")

print (f"Loaded {len(trainingData)} training and {len(validationData)} validation items")

Mounting drive
Mounted at /content/drive
Skipped (not a data set):  .ipynb_checkpoints
Loading:  anglechange-training-dw06e03p2.dat
Successfully loaded 4579 training items for video file dw06e03p2
Loading:  anglechange-validation-dw06e03p2.dat
Successfully loaded 644 validation items for video file dw06e03p2
Loaded 4579 training and 644 validation items
